In [1]:
import re
from sqlalchemy import create_engine
import urllib
import pandas as pd 

In [2]:
params_data = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=181.57.189.150,1433;"
    "DATABASE=Ventas_Shopify;"
    "UID=sa;"
    "PWD=P@ssw0rd*;"
)

# Crear el motor de conexión
engine_data = create_engine(f"mssql+pyodbc:///?odbc_connect={params_data}")

# Configurar cadena de conexión
params_com = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=181.57.189.150,1433;"
    "DATABASE=Ventas_Comerssia;"
    "UID=sa;"
    "PWD=P@ssw0rd*;"
)

# Crear el motor
engine_com = create_engine(f"mssql+pyodbc:///?odbc_connect={params_com}")

In [3]:
query_com = """
SELECT v.Cliente, 
	cc.nombres + ' ' + cc.Apellidos AS Nombre,
	cc.Email,
	cc.Celular,
	v.Fecha AS Fecha_Venta,
	v.Canal,
	SUM(Venta_Neta) AS Venta, 
	SUM(Descuento) AS Descuento
FROM Ventas_Comerssia.dbo.Ventas_Unificadas v
INNER JOIN Ventas_Comerssia.dbo.View_Contacto_Clientes cc ON v.Cliente = cc.Cliente
WHERE Codigo_Descuento IN ('0310 Clientes Cumple Julio Reactivo 25%','0309 Clientes Cumple Julio 2 Meses 25%','CUMPLE0825')
	AND Fecha BETWEEN '2025-08-01' AND '2025-08-31'
GROUP BY v.Cliente,
	cc.nombres + ' ' + cc.Apellidos,
	cc.Email,
	cc.Celular,
	v.Fecha,
	v.Canal
"""

# Ejecutar y cargar en DataFrame
df_ventas = pd.read_sql(query_com, engine_com)
df_ventas.head()

,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta,Descuento
0,C1000089807,PAULINA BERNAL POSADA,PAULINABERNALPOSADA@HOTMAIL.COM,3127739432,2025-08-12,Retail,34033.61,13500.0
1,C1000593985,JULIANA ROMERO,JULI.ROMERO13@HOTMAIL.COM,3106247450,2025-08-09,Retail,116596.64,46250.0
2,C1000794715,MARIA JOSE RODRIGUEZ JIMENEZ,MAJOROJIAMO15@HOTMAIL.COM,3172206929,2025-08-30,Shopify,289063.20,0.0
3,C1001090981,LUISA ARELLANO,LUISA_ARELLANO@HOTMAIL.COM,3017772461,2025-08-31,Retail,251470.59,99750.0
4,C10029397,JULIAN ESTRADA,JULESEST@HOTMAIL.COM,3166213281,2025-08-09,Shopify,432754.50,0.0


In [4]:
ruta_excel = 'Cumpleaños JULIO.xlsx'
df_plan_a = pd.read_excel(ruta_excel, sheet_name='PLAN A')
df_plan_b = pd.read_excel(ruta_excel, sheet_name='PLAN B ')

df_plan_a['Cliente'] = df_plan_a['Cliente'].astype(str)
df_plan_b['Cliente'] = df_plan_b['Cliente'].astype(str)


In [5]:
ventas_plan_a = df_ventas[df_ventas['Cliente'].isin(df_plan_a['Cliente'])]
total_clientes_a = ventas_plan_a['Cliente'].nunique()
total_ventas_a = ventas_plan_a['Venta'].sum()
total_descuento_a = ventas_plan_a['Descuento'].sum()

# 4. Filtrar ventas por clientes de Plan B
ventas_plan_b = df_ventas[df_ventas['Cliente'].isin(df_plan_b['Cliente'])]
total_clientes_b = ventas_plan_b['Cliente'].nunique()
total_ventas_b = ventas_plan_b['Venta'].sum()
total_descuento_b = ventas_plan_b['Descuento'].sum()

In [8]:
# 5. Mostrar resultados
print("Plan A:")
print(f"Clientes que compraron: {total_clientes_a}")
print(f"Total ventas: ${total_ventas_a:,.0f}")
print(f"Total descuento: ${total_descuento_a:,.0f}")

print("\nPlan B:")
print(f"Clientes que compraron: {total_clientes_b}")
print(f"Total ventas: ${total_ventas_b:,.0f}")
print(f"Total descuento: ${total_descuento_b:,.0f}")

Plan A:
Clientes que compraron: 95
Total ventas: $25,637,927
Total descuento: $9,260,750

Plan B:
Clientes que compraron: 117
Total ventas: $35,532,491
Total descuento: $13,424,250


In [11]:
total_ventas = df_ventas['Venta'].sum()
print(f"Total ventas: ${total_ventas:,.0f}")

Total ventas: $64,058,824
